# LA Studio – Direct Colab Voice Clone GPU

Run this notebook in a GPU Colab runtime. It starts a temporary, token-protected worker used directly by LA Studio Voice Cloning. The worker is independent from the API Gateway: do not enter a Gateway URL or key here.

Only clone a voice when you have the speaker's explicit permission and provide the exact reference transcript.

In [ ]:
import subprocess
import sys
from pathlib import Path

def run(*command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)

run('nvidia-smi')
run(sys.executable, '-m', 'pip', 'install', '--quiet', 'uv==0.8.13')

# Pinned worker release verified against LA Studio's direct /v2/jobs contract.
REPO_URL = 'https://github.com/khoinguyen59/kova-voice-studio.git'
REPO_REF = 'v1.0.2.1'
WORKSPACE = Path('/content/la-studio-voice-worker')
if not WORKSPACE.exists():
    run('git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(WORKSPACE))
else:
    run('git', 'fetch', '--depth', '1', 'origin', REPO_REF, cwd=WORKSPACE)
    run('git', 'checkout', '--force', REPO_REF, cwd=WORKSPACE)
    run('git', 'reset', '--hard', 'FETCH_HEAD', cwd=WORKSPACE)

VOICE_DIR = WORKSPACE / 'worker'
if not VOICE_DIR.is_dir():
    raise RuntimeError('Voice worker source is missing from the pinned release.')

run('uv', 'python', 'install', '3.11')
run('uv', 'venv', '--python', '3.11', '.venv', cwd=VOICE_DIR)
PYTHON = str(VOICE_DIR / '.venv' / 'bin' / 'python')

OMNIVOICE_REF = '0.2.1'
OMNIVOICE_DIR = Path('/content/OmniVoice')
if not OMNIVOICE_DIR.exists():
    run('git', 'clone', '--depth', '1', '--branch', OMNIVOICE_REF, 'https://github.com/k2-fsa/OmniVoice.git', str(OMNIVOICE_DIR))
else:
    run('git', 'fetch', '--depth', '1', 'origin', OMNIVOICE_REF, cwd=OMNIVOICE_DIR)
    run('git', 'checkout', '--force', OMNIVOICE_REF, cwd=OMNIVOICE_DIR)
    run('git', 'reset', '--hard', 'FETCH_HEAD', cwd=OMNIVOICE_DIR)

run('uv', 'pip', 'install', '--python', PYTHON, '--index-url', 'https://download.pytorch.org/whl/cu128', 'torch==2.8.0+cu128', 'torchaudio==2.8.0+cu128')
run('uv', 'pip', 'install', '--python', PYTHON, '-r', 'requirements-colab.txt', 'demucs==4.0.1', cwd=VOICE_DIR)
run('uv', 'pip', 'install', '--python', PYTHON, str(OMNIVOICE_DIR))
run('uv', 'pip', 'install', '--python', PYTHON, '-e', '.', cwd=VOICE_DIR)

doctor = subprocess.run([PYTHON, '-c', "import torch, transformers; assert torch.cuda.is_available(), 'CUDA is unavailable'; assert transformers.__version__ == '5.3.0', transformers.__version__; print(torch.cuda.get_device_name(0))"], text=True, capture_output=True)
print(doctor.stdout, end='')
if doctor.returncode:
    raise RuntimeError('CUDA dependency check failed. Reopen this pinned notebook and use a GPU runtime.\n' + doctor.stderr[-4000:])


In [ ]:
from pathlib import Path

# Keep LA Studio's standard capability catalog separate from the worker's own API.
# All profile and generation traffic remains direct to this one Colab process.
ENTRYPOINT = VOICE_DIR / 'la_studio_voice_clone_worker.py'
ENTRYPOINT.write_text(r'''
import os

import torch
from fastapi import FastAPI, Header, HTTPException
from kova_voice_studio.api import create_app

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Select a GPU runtime before starting this worker.')

TOKEN = os.environ['KOVA_VOICE_API_TOKEN']
worker_app = create_app()
app = FastAPI(title='LA Studio Direct Colab Voice Clone Worker', docs_url=None, redoc_url=None, openapi_url=None)

def require_token(authorization: str | None) -> None:
    if authorization != 'Bearer ' + TOKEN:
        raise HTTPException(status_code=401, detail='invalid worker token')

@app.get('/v1/capabilities')
def capabilities(authorization: str | None = Header(default=None)):
    require_token(authorization)
    return {
        'contract_version': 1,
        'device': 'cuda',
        'capabilities': [{
            'id': 'voice-cloning',
            'models': [{
                'id': 'omnivoice',
                'upstream_model': 'k2-fsa/OmniVoice',
                'formats': ['wav'],
                'reference_formats': ['wav', 'mp3', 'flac'],
                'reference_duration_seconds': {'min': 3, 'max': 30},
                'requires_consent': True,
                'loaded': True,
                'device': 'cuda',
            }],
        }],
    }

# The mounted worker owns /health, /v2/jobs/profile, /v2/jobs/generation,
# job status/cancellation/audio, and /v1/profiles.
app.mount('/', worker_app)
''', encoding='utf-8')
print(ENTRYPOINT)


In [ ]:
import json
import os
import re
import secrets
import signal
import subprocess
import time
import urllib.request

for pid_path in (Path('/content/la-studio-voice-clone-worker.pid'), Path('/content/la-studio-voice-clone-tunnel.pid')):
    try:
        pid = int(pid_path.read_text().strip())
        os.killpg(os.getpgid(pid), signal.SIGTERM)
    except (FileNotFoundError, ProcessLookupError, ValueError):
        pass
    finally:
        pid_path.unlink(missing_ok=True)
subprocess.run(['bash', '-lc', 'fuser -k 3923/tcp >/dev/null 2>&1 || true'], check=False)

TOKEN = secrets.token_urlsafe(32)
ENV = os.environ.copy()
ENV.update({
    'KOVA_VOICE_API_TOKEN': TOKEN,
    'KOVA_VOICE_REQUIRE_CUDA': '1',
    'KOVA_VOICE_PREPARE_PROFILE_PROMPT': '1',
    'KOVA_VOICE_DATA_DIR': '/content/la-studio-voice-clone-data',
})
LOG_PATH = Path('/content/la-studio-voice-clone-worker.log')
log_handle = LOG_PATH.open('w')
worker = subprocess.Popen([PYTHON, '-m', 'uvicorn', 'la_studio_voice_clone_worker:app', '--host', '127.0.0.1', '--port', '3923'], cwd=VOICE_DIR, env=ENV, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True)
Path('/content/la-studio-voice-clone-worker.pid').write_text(str(worker.pid))

for _ in range(60):
    try:
        request = urllib.request.Request('http://127.0.0.1:3923/health', headers={'Authorization': 'Bearer ' + TOKEN})
        with urllib.request.urlopen(request, timeout=3) as response:
            health = json.load(response)
        if health.get('device') == 'cuda':
            break
    except Exception:
        time.sleep(2)
else:
    worker.terminate()
    raise RuntimeError('Voice clone worker did not become CUDA-ready.\n' + LOG_PATH.read_text(errors='replace')[-4000:])

capability_request = urllib.request.Request('http://127.0.0.1:3923/v1/capabilities', headers={'Authorization': 'Bearer ' + TOKEN})
with urllib.request.urlopen(capability_request, timeout=10) as response:
    capabilities = json.load(response)
assert capabilities['capabilities'][0]['id'] == 'voice-cloning'
assert capabilities['capabilities'][0]['models'][0]['device'] == 'cuda'

CLOUDFLARED_VERSION = '2026.7.3'
CLOUDFLARED_SHA256 = '9d71c677db00134c1bd4144b7783486b654ad281b1ea62b4972098d19f770f17'
CLOUDFLARED = Path('/content/cloudflared')
subprocess.run(['wget', '-q', '-O', str(CLOUDFLARED), f'https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64'], check=True)
actual_hash = subprocess.check_output(['sha256sum', str(CLOUDFLARED)], text=True).split()[0]
if actual_hash != CLOUDFLARED_SHA256:
    raise RuntimeError('cloudflared checksum verification failed')
subprocess.run(['chmod', '+x', str(CLOUDFLARED)], check=True)
tunnel = subprocess.Popen([str(CLOUDFLARED), 'tunnel', '--url', 'http://127.0.0.1:3923', '--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)
Path('/content/la-studio-voice-clone-tunnel.pid').write_text(str(tunnel.pid))
public_url = None
for _ in range(90):
    line = tunnel.stdout.readline()
    print(line, end='')
    match = re.search(r'https://[^\s]+trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
if not public_url:
    worker.terminate()
    tunnel.terminate()
    raise RuntimeError('Cloudflare tunnel URL was not found')

print('\nLA_STUDIO_COLAB_VOICE_CLONE_URL=' + public_url)
print('LA_STUDIO_COLAB_VOICE_CLONE_TOKEN=' + TOKEN)
print('DEVICE=cuda  MODEL=omnivoice')
print('In LA Studio: Voice Cloning > settings > paste this URL and token > Use Colab GPU voice cloning.')
